# Base Model — Decoder Components Checklist

Transposed from `llm-from-scratch/src/working_with_text_data.ipynb`.

## Components
1. Tokenizer
2. Dataset & DataLoader
3. Token + Positional Embeddings
4. Self-Attention (v1 — raw parameters)
5. Self-Attention (v2 — `nn.Linear`)
6. Causal Attention (masked)
7. Multi-Head Attention Wrapper

## Imports

In [1]:
import re
import torch
import torch.nn as nn
import tiktoken
from torch.utils.data import Dataset, DataLoader
from importlib.metadata import version

print(f"torch version:    {version('torch')}")
print(f"tiktoken version: {version('tiktoken')}")
print(f"CUDA available:   {torch.cuda.is_available()}")

torch version:    2.9.0
tiktoken version: 0.9.0
CUDA available:   True


## 1 — Tokenizer

### 1.1 Load raw text

In [2]:
DATA_PATH = "/home/bmartins/dev/llm-from-scratch/data/the-verdict.txt"

with open(DATA_PATH, 'r') as f:
    raw_text = f.read()

print(f"Total characters: {len(raw_text)}")

Total characters: 20479


### 1.2 SimpleTokenizer (word-level)

In [3]:
class SimpleTokenizer:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i: s for s, i in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        preprocessed = [item if item in self.str_to_int else "<|unk|>" for item in preprocessed]
        return [self.str_to_int[s] for s in preprocessed]

    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        return re.sub(r'\s+([,.:;?_!"()\'])', r'\1', text)


# Build vocabulary
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]

all_tokens = sorted(set(preprocessed))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])
vocab = {token: i for i, token in enumerate(all_tokens)}

print(f"Vocabulary size: {len(vocab)}")

# Smoke test
tokenizer = SimpleTokenizer(vocab)
text = "It's the last he painted, you know, Mrs. Gisburn said with pardonable pride."
ids = tokenizer.encode(text)
print(f"Encoded: {ids}")
print(f"Decoded: {tokenizer.decode(ids)}")

Vocabulary size: 1132
Encoded: [56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 67, 7, 38, 851, 1108, 754, 793, 7]
Decoded: It' s the last he painted, you know, Mrs. Gisburn said with pardonable pride.


### 1.3 BPE Tokenizer via tiktoken (GPT-2)

In [4]:
tokenizer = tiktoken.get_encoding("gpt2")

text = "Hello, do you like tea? <|endoftext|> In the sunlit terraces of someunknownPlace."
integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
print(f"Encoded: {integers}")
print(f"Decoded: {tokenizer.decode(integers)}")

enc_text = tokenizer.encode(raw_text)
print(f"\nFull text token count: {len(enc_text)}")

Encoded: [15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 286, 617, 34680, 27271, 13]
Decoded: Hello, do you like tea? <|endoftext|> In the sunlit terraces of someunknownPlace.

Full text token count: 5145


### 1.4 Input → Target pair demonstration

In [5]:
enc_sample = enc_text[50:]
context_size = 4

for i in range(1, context_size + 1):
    context = enc_sample[:i]
    desired = enc_sample[i]
    print(f"{tokenizer.decode(context)!r:30s} ----> {tokenizer.decode([desired])!r}")

' and'                         ----> ' established'
' and established'             ----> ' himself'
' and established himself'     ----> ' in'
' and established himself in'  ----> ' a'


## 2 — Dataset & DataLoader

In [6]:
class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids  = []
        self.target_ids = []
        token_ids = tokenizer.encode(txt)
        for i in range(0, len(token_ids) - max_length, stride):
            self.input_ids.append(torch.tensor(token_ids[i:i + max_length]))
            self.target_ids.append(torch.tensor(token_ids[i + 1:i + max_length + 1]))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]


def create_dataloader_v1(txt, batch_size=4, max_length=256, stride=128,
                         shuffle=True, drop_last=True, num_workers=0):
    tok = tiktoken.get_encoding("gpt2")
    dataset = GPTDatasetV1(txt, tok, max_length, stride)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle,
                      drop_last=drop_last, num_workers=num_workers)


# Smoke test
dataloader = create_dataloader_v1(raw_text, batch_size=8, max_length=4, stride=4, shuffle=False)
inputs, targets = next(iter(dataloader))

print(f"inputs shape:  {inputs.shape}")
print(f"targets shape: {targets.shape}")
print(f"inputs:\n{inputs}")
print(f"targets:\n{targets}")

inputs shape:  torch.Size([8, 4])
targets shape: torch.Size([8, 4])
inputs:
tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])
targets:
tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])


## 3 — Token + Positional Embeddings

In [7]:
VOCAB_SIZE = 50257
OUTPUT_DIM = 256
MAX_LENGTH = 4

token_embedding_layer = nn.Embedding(VOCAB_SIZE, OUTPUT_DIM)
pos_embedding_layer   = nn.Embedding(MAX_LENGTH, OUTPUT_DIM)

dataloader = create_dataloader_v1(
    raw_text, batch_size=8, max_length=MAX_LENGTH, stride=MAX_LENGTH, shuffle=False
)
inputs, targets = next(iter(dataloader))

token_embeddings = token_embedding_layer(inputs)
pos_embeddings   = pos_embedding_layer(torch.arange(MAX_LENGTH))
input_embeddings = token_embeddings + pos_embeddings

print(f"token_embeddings shape: {token_embeddings.shape}   [batch, seq_len, emb_dim]")
print(f"pos_embeddings shape:   {pos_embeddings.shape}   [seq_len, emb_dim]")
print(f"input_embeddings shape: {input_embeddings.shape}   [batch, seq_len, emb_dim]")

token_embeddings shape: torch.Size([8, 4, 256])   [batch, seq_len, emb_dim]
pos_embeddings shape:   torch.Size([4, 256])   [seq_len, emb_dim]
input_embeddings shape: torch.Size([8, 4, 256])   [batch, seq_len, emb_dim]


## 4 — Self-Attention v1 (raw `nn.Parameter`)

Simplest formulation: W_query, W_key, W_value as raw parameter matrices.
No masking — every token attends to every other token (bidirectional).

In [8]:
class SelfAttention_v1(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()
        self.W_query = nn.Parameter(torch.rand(d_in, d_out))
        self.W_key   = nn.Parameter(torch.rand(d_in, d_out))
        self.W_value = nn.Parameter(torch.rand(d_in, d_out))

    def forward(self, x):
        keys    = x @ self.W_key
        queries = x @ self.W_query
        values  = x @ self.W_value
        attn_scores  = queries @ keys.T
        attn_weights = torch.softmax(attn_scores / keys.shape[-1] ** 0.5, dim=-1)
        return attn_weights @ values


# 6-token sequence, 3-dim embeddings -> project to 2-dim
inputs = torch.tensor([
    [0.43, 0.15, 0.89],
    [0.55, 0.87, 0.66],
    [0.57, 0.85, 0.64],
    [0.22, 0.58, 0.33],
    [0.77, 0.25, 0.10],
    [0.05, 0.80, 0.55]
])

torch.manual_seed(123)
sa_v1 = SelfAttention_v1(d_in=3, d_out=2)
print(f"SelfAttention_v1 output:\n{sa_v1(inputs)}")

SelfAttention_v1 output:
tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]], grad_fn=<MmBackward0>)


## 5 — Self-Attention v2 (`nn.Linear`)

Replaces raw `nn.Parameter` matrices with `nn.Linear` — same math, better initialisation and weight management.

In [9]:
class SelfAttention_v2(nn.Module):
    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

    def forward(self, x):
        keys    = self.W_key(x)
        queries = self.W_query(x)
        values  = self.W_value(x)
        attn_scores  = queries @ keys.T
        attn_weights = torch.softmax(attn_scores / keys.shape[-1] ** 0.5, dim=-1)
        return attn_weights @ values


torch.manual_seed(789)
sa_v2 = SelfAttention_v2(d_in=3, d_out=2)
print(f"SelfAttention_v2 output:\n{sa_v2(inputs)}")

SelfAttention_v2 output:
tensor([[-0.0739,  0.0713],
        [-0.0748,  0.0703],
        [-0.0749,  0.0702],
        [-0.0760,  0.0685],
        [-0.0763,  0.0679],
        [-0.0754,  0.0693]], grad_fn=<MmBackward0>)


## 6 — Causal Attention (masked)

Adds an upper-triangular mask so each token can only attend to itself and tokens before it.
This is what makes the Transformer **decoder-only** (autoregressive).

```
mask[i, j] = -inf  if j > i   (future token — blocked)
mask[i, j] = 0     if j <= i  (past/current — allowed)
```

In [10]:
class CausalAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, qkv_bias=False):
        super().__init__()
        self.d_out   = d_out
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        keys    = self.W_key(x)
        queries = self.W_query(x)
        values  = self.W_value(x)
        attn_scores = queries @ keys.transpose(1, 2)
        attn_scores.masked_fill_(self.mask.bool()[:num_tokens, :num_tokens], -torch.inf)
        attn_weights = torch.softmax(attn_scores / keys.shape[-1] ** 0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)
        return attn_weights @ values


# batch of 2 identical sequences (6 tokens, 3-dim)
batch = torch.stack((inputs, inputs), dim=0)   # [2, 6, 3]

torch.manual_seed(123)
context_length = batch.shape[1]
ca = CausalAttention(d_in=3, d_out=2, context_length=context_length, dropout=0.0)
context_vecs = ca(batch)

print(f"Input shape:  {batch.shape}")
print(f"Output shape: {context_vecs.shape}")
print(f"Context vectors:\n{context_vecs}")

Input shape:  torch.Size([2, 6, 3])
Output shape: torch.Size([2, 6, 2])
Context vectors:
tensor([[[-0.4519,  0.2216],
         [-0.5874,  0.0058],
         [-0.6300, -0.0632],
         [-0.5675, -0.0843],
         [-0.5526, -0.0981],
         [-0.5299, -0.1081]],

        [[-0.4519,  0.2216],
         [-0.5874,  0.0058],
         [-0.6300, -0.0632],
         [-0.5675, -0.0843],
         [-0.5526, -0.0981],
         [-0.5299, -0.1081]]], grad_fn=<UnsafeViewBackward0>)


## 7 — Multi-Head Attention Wrapper

Runs `num_heads` independent `CausalAttention` heads and concatenates their outputs.
Each head projects to `d_out` dims; concatenated output is `num_heads * d_out` dims.

> **Note:** This is the naive wrapper approach (one full set of QKV weights per head).
> The efficient single-matrix implementation lives in `GPTModel.ipynb` as `MultiHeadAttention`.

In [11]:
class MultiHeadAttentionWrapper(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        self.heads = nn.ModuleList([
            CausalAttention(d_in, d_out, context_length, dropout, qkv_bias)
            for _ in range(num_heads)
        ])

    def forward(self, x):
        return torch.cat([head(x) for head in self.heads], dim=-1)


context_length = batch.shape[1]   # 6 tokens
mha = MultiHeadAttentionWrapper(
    d_in=3, d_out=2, context_length=context_length, dropout=0.0, num_heads=2
)
context_vecs = mha(batch)

print(f"Input shape:  {batch.shape}          [batch, seq_len, d_in]")
print(f"Output shape: {context_vecs.shape}   [batch, seq_len, num_heads * d_out]")
print(f"\nContext vectors:\n{context_vecs}")
print()
print("dim 0 = 2 sequences (duplicated input)")
print("dim 1 = 6 tokens per sequence")
print("dim 2 = 4 = 2 heads x 2 dims per head")

Input shape:  torch.Size([2, 6, 3])          [batch, seq_len, d_in]
Output shape: torch.Size([2, 6, 4])   [batch, seq_len, num_heads * d_out]

Context vectors:
tensor([[[0.4772, 0.1063, 0.4566, 0.2729],
         [0.5891, 0.3257, 0.5792, 0.3011],
         [0.6202, 0.3860, 0.6249, 0.3102],
         [0.5478, 0.3589, 0.5691, 0.2785],
         [0.5321, 0.3428, 0.5543, 0.2520],
         [0.5077, 0.3493, 0.5337, 0.2499]],

        [[0.4772, 0.1063, 0.4566, 0.2729],
         [0.5891, 0.3257, 0.5792, 0.3011],
         [0.6202, 0.3860, 0.6249, 0.3102],
         [0.5478, 0.3589, 0.5691, 0.2785],
         [0.5321, 0.3428, 0.5543, 0.2520],
         [0.5077, 0.3493, 0.5337, 0.2499]]], grad_fn=<CatBackward0>)

dim 0 = 2 sequences (duplicated input)
dim 1 = 6 tokens per sequence
dim 2 = 4 = 2 heads x 2 dims per head


## Checklist Summary

| Component | Class | Status |
|---|---|---|
| BPE Tokenizer | `tiktoken` (GPT-2) | ✓ |
| Dataset (sliding window) | `GPTDatasetV1` | ✓ |
| DataLoader | `create_dataloader_v1` | ✓ |
| Token embeddings | `nn.Embedding` | ✓ |
| Positional embeddings | `nn.Embedding` | ✓ |
| Self-attention (raw params) | `SelfAttention_v1` | ✓ |
| Self-attention (Linear) | `SelfAttention_v2` | ✓ |
| Causal mask | `CausalAttention` | ✓ |
| Multi-head attention | `MultiHeadAttentionWrapper` | ✓ |
| Layer Norm | — | → `GPTModel.ipynb` |
| GELU activation | — | → `GPTModel.ipynb` |
| Feed-Forward Network | — | → `GPTModel.ipynb` |
| Transformer Block | — | → `GPTModel.ipynb` |
| Full GPT Model | — | → `GPTModel.ipynb` |